## 2. 웹스크래핑 연습문제

2-1. Nate 뉴스기사 제목 스크래핑하기 (필수) 

In [7]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import Image, display

# 1. 섹션명과 mid 매핑 딕셔너리
section_dict = {
    '최신뉴스': 'n0100',
    '정치': 'n0200',
    '경제': 'n0300',
    '사회': 'n0400',
    '세계': 'n0500',
    'IT/과학': 'n0600'
}

def print_nate_news(section_name):
    mid = section_dict.get(section_name)
    if not mid:
        print(f"'{section_name}'은(는) 존재하지 않는 섹션입니다.")
        return

    url = f'https://news.nate.com/recent?mid={mid}'
    
    # 브라우저처럼 보이기 위한 헤더 설정
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    res = requests.get(url, headers=headers)
    if not res.ok:
        print(f"페이지를 불러오지 못했습니다. 에러 코드: {res.status_code}")
        return

    soup = BeautifulSoup(res.text, 'html.parser')
    
    print(f"\n{'='*20}> 네이트 [{section_name}] 뉴스 <{'='*20}\n")

    # 뉴스 아이템 선택 (네이트 최신뉴스의 경우 .post-item 구조를 주로 가짐)
    articles = soup.select('div.post-item')

    for article in articles:
        # 제목 및 링크 추출
        title_tag = article.select_one('span.tit')
        a_tag = article.select_one('a')
        
        if title_tag and a_tag:
            title = title_tag.text.strip()
            link = urljoin(url, a_tag['href'])
            
            # 이미지 엘리먼트 존재 여부 체크
            img_tag = article.select_one('img')
            
            if img_tag and 'src' in img_tag.attrs:
                src = img_tag['src']
                # urljoin을 사용하여 //로 시작하거나 상대 경로인 주소를 절대 경로로 변환
                img_url = urljoin(url, src)
                # 이미지 출력
                display(Image(url=img_url, width=150))
            else:
                print("[이미지 없음]")

            print(f"기사제목: {title}")
            print(f"기사링크: {link}")
            print("-" * 60)

# 함수 호출 예시
print_nate_news('IT/과학')


====================> 네이트 [IT/과학] 뉴스 <====================



2-2. 하나의 네이버 웹툰과 1개의 회차에 대한 Image 다운로드 하기 (필수) 

In [ ]:
import os
import requests
from bs4 import BeautifulSoup

def download_one_episode(title, no, url):
    # 1. 디렉토리 생성 (img\제목\회차번호)
    save_path = os.path.join('img', title, str(no))
    os.makedirs(save_path, exist_ok=True)
    
    # 2. 웹툰 페이지 요청 헤더 설정 
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': url
    }

    res = requests.get(url, headers=headers)
    if not res.ok:
        print(f"웹툰 페이지 접속 실패: {res.status_code}")
        return

    soup = BeautifulSoup(res.text, 'html.parser')

    # 3. 웹툰 이미지 태그들 찾기
    img_tags = soup.select("img[src*='IMAG01']")
    
    if not img_tags:
        print("이미지를 찾을 수 없습니다. 선택자를 확인해 주세요.")
        return

    print(f"[{title}] {no}화 다운로드 시작...")

    # 4. 이미지 다운로드 및 저장
    for i, img_tag in enumerate(img_tags):
        img_url = img_tag['src']
        
        # 이미지 데이터 요청
        img_res = requests.get(img_url, headers=headers)
        
        if img_res.ok:
            img_data = img_res.content
            # 파일명 생성 
            file_name = f"{i:03d}_{os.path.basename(img_url.split('?')[0])}"
            file_full_path = os.path.join(save_path, file_name)
            
            # 바이너리 모드로 저장
            with open(file_full_path, 'wb') as f:
                f.write(img_data)
            print(f"저장 완료: {file_name} ({len(img_data):,} bytes)")
        else:
            print(f"이미지 다운로드 실패: {img_url}")

    print(f"\n✅ '{title}' {no}화 모든 이미지 다운로드 완료!")

# 함수 호출 예시
download_one_episode('괴물 천재선수들이 날 너무 좋아함', 9, 'https://comic.naver.com/webtoon/detail?titleId=843901&no=9&week=sun')

[일렉시드] 341화 다운로드 시작...
저장 완료: 000_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_1.jpg (87,143 bytes)
저장 완료: 001_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_2.jpg (256,127 bytes)
저장 완료: 002_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_3.jpg (184,536 bytes)
저장 완료: 003_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_4.jpg (182,867 bytes)
저장 완료: 004_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_5.jpg (112,615 bytes)
저장 완료: 005_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_6.jpg (169,889 bytes)
저장 완료: 006_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_7.jpg (157,876 bytes)
저장 완료: 007_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_8.jpg (181,837 bytes)
저장 완료: 008_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_9.jpg (203,632 bytes)
저장 완료: 009_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_10.jpg (113,543 bytes)
저장 완료: 010_20250311184953_812848a288eb3e6ce6efe904abd0ef68_IMAG01_11.jpg (166,818 bytes)
저장 완료: 0

2-3. 하나의 네이버 웹툰과 여러개의 회차에 대한 Image 다운로드 하기 (선택)